In [1]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
import pickle


In [2]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\features.csv")
print(df.shape)
df.head()

(32101, 19)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,result,home_elo,away_elo,elo_diff,home_conf,away_conf,home_fifa_rank,away_fifa_rank,fifa_rank_diff
0,1990-01-12,Algeria,Mali,5,0,Friendly,Paris,France,True,0.5,H,1837.945150,1646.944724,191.000426,CAF,CAF,28.0,25.0,3.0
1,1990-01-14,Algeria,Cameroon,3,1,Friendly,Paris,France,True,0.5,H,1837.945150,1731.465033,106.480117,CAF,CAF,28.0,25.0,3.0
2,1990-01-17,Greece,Belgium,2,0,Friendly,Athens,Greece,False,0.5,H,1726.773470,1780.811104,-54.037634,UEFA,UEFA,25.0,9.0,16.0
3,1990-01-17,Mexico,Argentina,2,0,Friendly,Los Angeles,United States,True,0.5,H,1911.874693,2047.584193,-135.709501,CONCACAF,CONMEBOL,15.0,3.0,12.0
4,1990-01-20,Malawi,Tanzania,2,2,Friendly,Lobamba,Swaziland,True,0.5,D,1384.462883,1388.469275,-4.006393,CAF,CAF,25.0,25.0,0.0


In [3]:
# Drop rows with any missing values in key columns
model_df = df.dropna(subset=[
    'home_score', 'away_score',
    'home_elo', 'away_elo',
    'elo_diff', 'fifa_rank_diff',
    'tournament_weight', 'neutral'
]).copy()

# Convert neutral to integer (True=1, False=0)
model_df['neutral'] = model_df['neutral'].astype(int)

print(f"Training rows: {len(model_df)}")

Training rows: 32101


In [4]:
home_model = glm(
    'home_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight + fifa_rank_diff',
    data=model_df,
    family=Poisson()
).fit()

print(home_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_score   No. Observations:                32101
Model:                            GLM   Df Residuals:                    32095
Model Family:                 Poisson   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -52338.
Date:                Mon, 08 Jun 2026   Deviance:                       44045.
Time:                        15:32:44   Pearson chi2:                 4.38e+04
No. Iterations:                    37   Pseudo R-squ. (CS):             0.2582
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             1.0412      0.03

In [5]:
away_model = glm(
    'away_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight + fifa_rank_diff',
    data=model_df,
    family=Poisson()
).fit()

print(away_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             away_score   No. Observations:                32101
Model:                            GLM   Df Residuals:                    32095
Model Family:                 Poisson   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -43959.
Date:                Mon, 08 Jun 2026   Deviance:                       41305.
Time:                        15:33:10   Pearson chi2:                 4.14e+04
No. Iterations:                   100   Pseudo R-squ. (CS):             0.2085
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.3506      0.04

In [6]:
import pickle

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\home_model.pkl", 'wb') as f:
    pickle.dump(home_model, f)

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\away_model.pkl", 'wb') as f:
    pickle.dump(away_model, f)

print("Both models saved!")

Both models saved!


In [7]:
# Spain vs Argentina on neutral ground
test_match = pd.DataFrame([{
    'home_elo': 2169.69,   # Spain
    'away_elo': 2047.58,   # Argentina
    'elo_diff': 2169.69 - 2047.58,
    'neutral': 1,
    'tournament_weight': 5.0,
    'fifa_rank_diff': 2 - 3   # Spain rank 2, Argentina rank 3
}])

home_goals = home_model.predict(test_match)[0]
away_goals = away_model.predict(test_match)[0]

print(f"Spain expected goals:     {home_goals:.2f}")
print(f"Argentina expected goals: {away_goals:.2f}")

Spain expected goals:     1.38
Argentina expected goals: 0.92


In [8]:
from scipy.stats import poisson

def match_probabilities(home_xg, away_xg, max_goals=10):
    home_win = 0
    draw = 0
    away_win = 0

    for i in range(max_goals):
        for j in range(max_goals):
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            if i > j:
                home_win += p
            elif i == j:
                draw += p
            else:
                away_win += p

    return round(home_win, 4), round(draw, 4), round(away_win, 4)

home_win, draw, away_win = match_probabilities(home_goals, away_goals)

print(f"Spain win:     {home_win*100:.1f}%")
print(f"Draw:          {draw*100:.1f}%")
print(f"Argentina win: {away_win*100:.1f}%")

Spain win:     47.7%
Draw:          27.4%
Argentina win: 24.9%


In [9]:
feature_cols = ['home_elo', 'away_elo', 'elo_diff', 'neutral', 'tournament_weight', 'fifa_rank_diff']

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\feature_cols.pkl", 'wb') as f:
    pickle.dump(feature_cols, f)

print("Feature columns saved!")

Feature columns saved!
